# distill_safety — Colab launcher

This notebook is a **thin launcher**: all logic lives in the repo. Every stage writes checkpoints / shards / `.done` markers to Google Drive, so after a disconnect just **re-run all cells** and the pipeline resumes.

Runtime → Change runtime type → **A100** (7B teacher) or **L4** (≤1.5B jobs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = 'https://github.com/AlbertStephen/distill_safety.git'
DRIVE_ROOT = '/content/drive/MyDrive/distill_safety'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists('/content/distill_safety'):
    !git clone -q $REPO_URL /content/distill_safety
else:
    !git -C /content/distill_safety pull -q
%cd /content/distill_safety
!pip install -q -r requirements-colab.txt
# cache HF models on Drive so they are not re-downloaded every session
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'

In [ ]:
# Optional: experiment tracking
# import wandb; wandb.login()
# First pilot run on A100 (~20-40 min): verify 7B teacher ASR >= 90% and Drive resume.
# For full 40k run, set OVERRIDES = 'distill.trigger_exposure=0.01'
CONFIG = 'configs/default_7b.yaml'
RUN_NAME = 'pilot_7b_rare_r0.05'
OVERRIDES = 'teacher.n_train=2000 distill.n_prompts=5000 distill.trigger_exposure=0.05 eval.n_eval=200'

In [ ]:
!python scripts/run_pipeline.py --config $CONFIG run_name=$RUN_NAME output_root=$DRIVE_ROOT/outputs $OVERRIDES

In [ ]:
import json
print(json.dumps(json.load(open(f'{DRIVE_ROOT}/outputs/{RUN_NAME}/summary.json')), indent=2))